In [20]:
"""
Dependancy Imports
"""
import yaml
import numpy as np
import sys
import os

# Adds the parent directory of 'src' to the system path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))


def load_config(path="../configs/default.yaml"):
  with open(path, "r") as f:
    return yaml.safe_load(f)
cfg = load_config()

### Implementing the path loss and steering vector
Eq 12. Get the path loss
$$
\beta(d) = \beta_0 \left( \frac{d}{d_0} \right)^{-\eta} 
$$

Eq 11. Get the ULA steering vector

$$
a_Q(\theta) = \frac{1}{\sqrt{Q}} \left[ 1, e^{j \pi \sin \theta}, \dots, e^{j \pi (Q-1) \sin \theta} \right]^T
$$

In [27]:
from src.utils.channel_utils import to_db, to_linear

def lamba():
  return float(cfg["channel_model"]["c"]) / float(cfg["channel_model"]["carrier_frequency"])

def steering_vector(elements: int, angle_rad: float) -> np.ndarray:
  q = np.arange(elements)
  return (1.0 / np.sqrt(elements)) * np.exp(1j * np.pi * q * np.sin(angle_rad))


def get_path_loss_linear(d: float, eta:float):
  return to_linear(int(cfg["channel_model"]["beta_0_dB"])) * ((d / int(cfg["channel_model"]["d_0"])) ** -eta)

### Implement Channel Models
- Rician Fading
- Rayleigh Fading

Equation 10 and 11 implemented

In [28]:
def get_H_bar(rx_elements: int, tx_elements: int, angle_rx_rad: float, angle_tx_rad: float = None) -> np.ndarray:
  a_rx = steering_vector(rx_elements, angle_rx_rad)
  if tx_elements == 1:
    return a_rx.reshape(-1,1)
  
  a_tx = steering_vector(tx_elements, angle_tx_rad)
  return np.outer(a_rx, a_tx.conj())

def get_H_tilde(rx_elements: int, tx_elements: int, rng: np.random.Generator) -> np.ndarray:
  real = rng.standard_normal((rx_elements, tx_elements))
  imag = rng.standard_normal((rx_elements, tx_elements))
  return (real + 1j * imag) / np.sqrt(2.0)

def get_hybird_channel_model(rx_elements:int, tx_elements:int, rician_factor: int, beta:float, angle_rx_rad: float, angle_tx_rad: float = None, rng: np.random.Generator = None) -> np.ndarray:
  h_bar = get_H_bar(rx_elements=rx_elements, tx_elements=tx_elements, 
  angle_rx_rad=angle_rx_rad, angle_tx_rad=angle_tx_rad)
  h_tilde = get_H_tilde(rx_elements=rx_elements, tx_elements=tx_elements, rng=rng)

  scale_h_bar_to_path_loss_power = (np.sqrt(beta * rician_factor) / rician_factor + 1) * h_bar

  scale_h_tilde_to_path_loss_pwoer = (np.sqrt(beta / (rician_factor + 1))) * h_tilde

  return scale_h_bar_to_path_loss_power + scale_h_tilde_to_path_loss_pwoer


In [32]:
from src.utils.channel_utils import to_db, to_linear
rng = np.random.default_rng(0)
rx = 32
tx = 8
beta = get_path_loss_linear(d = 50.0, eta = cfg["channel_model"]["eta_los"])
rician_factor = to_linear(5.0)

channel = get_hybird_channel_model(rx,tx,rician_factor,beta,0.3, 0.2, rng)
print("channel shape:", channel.shape, "| avg power:", to_db(np.mean(np.abs(channel) ** 2)))
 

channel shape: (32, 8) | avg power: -24.082364683150367
